In [1]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
llm = ChatOllama(model="llama3.1", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def reasoning_node(state: AgentState):
    """This function acts as the 'brain' node in our graph."""
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]}

graph_builder = StateGraph(AgentState)
graph_builder.add_node("reasoning", reasoning_node)
graph_builder.add_edge(START, "reasoning")
graph_builder.add_edge("reasoning", END)
agent = graph_builder.compile()

from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
import json

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together. You MUST use this for math."""
    return a * b

tools = [multiply]

llm_with_tools = llm.bind_tools(tools)

def safe_tool_node(state: AgentState):
    """Executes the tool, but catches errors and feeds them back to the LLM."""
    last_message = state['messages'][-1]
    if not getattr(last_message, 'tool_calls', None):
        return {"messages": []}
    tool_call = last_message.tool_calls[0]
    try:
        available_tools = {"multiply": multiply}
        chosen_tool = available_tools[tool_call["name"]]
        result = chosen_tool.invoke(tool_call["args"])
        return {"messages": [ToolMessage(content=str(result), tool_call_id=tool_call["id"])]}
    except Exception as e:
        error_msg = f"Error executing tool. Please fix your formatting and try again. Error: {str(e)}"
        print(f"⚠️ Caught an error! Feeding back to LLM: {error_msg}")
        
        return {"messages": [ToolMessage(content=error_msg, tool_call_id=tool_call["id"])]}

print("Step 1 Complete: Safe Tool Node defined with error-catching guardrails!")

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 1 Complete: Safe Tool Node defined with error-catching guardrails!


Right now, our graph is a straight line. But to be an agent, the model needs to act, observe the result, and act again. We need a loop.

However, local models are notorious for the "Infinite Ping-Pong of Doom."
1. The LLM formats a tool call poorly.
2. The Tool Node catches the error and sends it back.
3. The LLM panics, ignores the error message, and sends the exact same bad tool call.
4. Repeat until your computer melts.

In [2]:
from langgraph.graph import StateGraph, START, END

# 1. Redefine our Reasoning Node to use the LLM that knows about our tools
# (We created `llm_with_tools` in the last step)
def reasoning_node_with_tools(state: AgentState):
    """The brain, now equipped with tools."""
    response = llm_with_tools.invoke(state['messages'])
    return {"messages": [response]}

# 2. The Router (Conditional Edge)
def should_continue(state: AgentState):
    """Looks at the LLM's last message to decide where to go next."""
    last_message = state['messages'][-1]
    
    # If there are no tool calls, the LLM is done thinking. Route to END.
    if not getattr(last_message, 'tool_calls', None):
        return "end"
        
    # If there IS a tool call, route to the tool node.
    return "continue"

# 3. Rebuild the Graph
workflow = StateGraph(AgentState)

# Add our two workers
workflow.add_node("reasoning", reasoning_node_with_tools)
workflow.add_node("tools", safe_tool_node)

# 4. Draw the Loop Map
workflow.add_edge(START, "reasoning")

# Instead of a straight line to END, we add a fork in the road
workflow.add_conditional_edges(
    "reasoning",      # After 'reasoning' runs...
    should_continue,  # ...pass the state to this routing function...
    {
        "continue": "tools", # ...if it returns "continue", go to tools
        "end": END           # ...if it returns "end", we are finished
    }
)

# Crucial Step: After the tools finish, ALWAYS route back to the brain!
workflow.add_edge("tools", "reasoning")

# Compile the graph
agent = workflow.compile()

print("Step 2 & 3 Complete: Loop mapped and graph compiled!")

Step 2 & 3 Complete: Loop mapped and graph compiled!


In [ ]:
# 1. Define the input state with a math question
initial_state = {"messages": [("user", "What is 123 multiplied by 456?")]}

# 2. Set the strict recursion limit (The Circuit Breaker)
# This tells LangGraph to forcefully stop if the graph transitions more than 5 times
config = {"recursion_limit": 5}

print("Running the autonomous loop...\n")

# 3. Stream the graph execution as a spectator
try:
    # You drop the user's message ("What is 123 multiplied by 456?") into the empty AgentState clipboard. 
    # LangGraph looks at your blueprint, sees the START edge, and routes the clipboard directly to the reasoning node.
    for event in agent.stream(initial_state, config=config):
        for node_name, node_state in event.items():
            # Turn 1: reasoning_node_with_tools
                # LLM reads the state, realises there is a tool and a request for math and returns a JSON block
                # LangChain converts this into an AIMessage with a .tool_calls attribute.
                # LangGraph appends this new message to the clipboard. The reasoning node's turn ends, and it yields its event to your console.
            print(f"--- Output from node: {node_name} ---")
            # Grab the last message to see what just happened
            last_message = node_state['messages'][-1]
            # LangGraph hits our conditional edge. It passes the clipboard to our should_continue function.
            # The function looks at the very last message on the clipboard. It sees the .tool_calls attribute. -> function ouputs "continue"
            # Turn 2: safe_tool_node
                # Our tool node opens the clipboard, finds the tool call, and extracts the numbers 123 and 456.
                # It runs our plain Python multiply function, which calculates 56088.
                # It wraps 56088 in a ToolMessage and appends it to the bottom of the clipboard. 
                # The tools node finishes and yields its event to the console.
            # Check if it was a tool call, a tool result, or final text
            if getattr(last_message, 'tool_calls', None):
                print(f"🛠️ LLM decided to call a tool: {last_message.tool_calls[0]['name']}")
                print(f"Arguments: {last_message.tool_calls[0]['args']}")
            elif node_name == "tools":
                print(f"✅ Tool returned result: {last_message.content}")
            # Because of workflow.add_edge("tools", "reasoning"), it forcefully routes the updated clipboard back to the reasoning node.
            else:
                # Turn 3 - The Brain... Again (reasoning_node)
                    # The LLM reads the clipboard again. Now the history looks like this:
                        # User: What is 123 * 456?
                        # LLM: Attempted to use multiply
                        # Tool: The result is 56088.
                        # The LLM sees the tool gave it the answer! Now, it generates standard conversational text: "123 multiplied by 456 is 56,088."
                print(f"🧠 LLM says: {last_message.content}\n")

It appends this final message to the clipboard and finishes its turn.
                
except Exception as e:
    # If the model gets stuck in an infinite loop, LangGraph throws a GraphRecursionError
    print(f"\n🚨 CIRCUIT BREAKER TRIPPED! The agent got stuck. Error: {e}")

print("\nModule 2 Complete! You have successfully wrangled the loop.")

Running the autonomous loop...

--- Output from node: reasoning ---
🛠️ LLM decided to call a tool: multiply
Arguments: {'a': '123', 'b': '456'}
--- Output from node: tools ---
✅ Tool returned result: 56088
--- Output from node: reasoning ---
🧠 LLM says: The result of multiplying 123 by 456 is 56,088.


Module 2 Complete! You have successfully wrangled the loop.
